In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
df = pd.read_csv("/content/final_df.csv")

In [6]:
skill_columns = {
    'Skills - Python': 'Python',
    'Skills - SQL': 'SQL',
    'Skills - ML': 'MachineLearning',
    'Skills-DeepLearning': 'DeepLearning',
    'skills_Cloud': 'Cloud'
}

In [7]:
def create_skill_text(row):

    skills = []

    for column, skill_name in skill_columns.items():

        if row[column] == 1:
            skills.append(skill_name)

    return " ".join(skills)


# Create new column
df['skills_text'] = df.apply(create_skill_text, axis=1)

In [8]:
vectorizer = TfidfVectorizer()

job_skill_vectors = vectorizer.fit_transform(df['skills_text'])

In [9]:
joblib.dump(vectorizer, "new_tfidf_vectorizer.pkl")
joblib.dump(job_skill_vectors, "new_job_skill_vectors.pkl")
joblib.dump(df, "new_jobs_data.pkl")

['new_jobs_data.pkl']

In [11]:
import joblib
from sklearn.metrics.pairwise import cosine_similarity


# Load exported files
vectorizer = joblib.load("new_tfidf_vectorizer.pkl")
job_vectors = joblib.load("new_job_skill_vectors.pkl")
df = joblib.load("new_jobs_data.pkl")

# Recommendation Function
def recommend_jobs(user_skills, top_n=5):

    # Convert list into text
    user_text = " ".join(user_skills)

    # Convert into vector
    user_vector = vectorizer.transform([user_text])

    # Similarity score
    similarity_scores = cosine_similarity(user_vector, job_vectors)

    # Get top indexes
    top_indexes = similarity_scores.argsort()[0][-top_n:][::-1]

    # Fetch recommended jobs
    recommendations = df.iloc[top_indexes][
        ['JobTitle', 'company_Industry', 'Salary (USD)']
    ]

    return recommendations


In [12]:
user_skills = [
    'Python',
    'SQL',
    'MachineLearning'
]

results = recommend_jobs(user_skills)

print(results)

                       JobTitle company_Industry  Salary (USD)
9970             Data Scientist               IT     212076.75
564                Data Analyst       E-commerce     111100.00
6943  Machine Learning Engineer          Unknown     134201.00
6914  Machine Learning Engineer          Finance     179592.00
6917             Data Scientist           Retail     131567.00
